# Credit Risk Analytics & Risk Modelling

**Educational portfolio project.** The included dataset is synthetic and the target labels are artificially generated. Metrics are not evidence of real-world credit-risk performance.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "credit_risk_synthetic.csv").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "credit_risk_synthetic.csv"
df = pd.read_csv(DATA_PATH)
df.head()


## 1. Data quality and overview

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.isna().sum().to_frame("missing_values"))
display(df.describe(include="all").T)


## 2. Target distribution

In [ ]:
ax = sns.countplot(data=df, x="default")
ax.set(title="Synthetic target distribution", xlabel="Demo default label (0/1)", ylabel="Record count")
plt.show()
print("Synthetic positive-label rate:", round(df["default"].mean(), 3))


## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data=df, x="annual_income", kde=True, ax=axes[0])
axes[0].set_title("Annual income (synthetic)")
sns.histplot(data=df, x="credit_score", kde=True, ax=axes[1])
axes[1].set_title("Credit-score-like value (synthetic)")
sns.boxplot(data=df, x="default", y="debt_to_income", ax=axes[2])
axes[2].set_title("Debt-to-income by demo label")
plt.tight_layout()
plt.show()


In [ ]:
numeric = df.select_dtypes(include="number")
plt.figure(figsize=(9, 6))
sns.heatmap(numeric.drop(columns=["applicant_id"], errors="ignore").corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric feature correlations (synthetic data)")
plt.tight_layout()
plt.show()


## 4. Baseline modelling

This section uses a stratified holdout split and preprocessing pipelines. The scores are for workflow demonstration only because the labels are synthetic.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns=["default", "applicant_id"])
y = df["default"]
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")
}


In [ ]:
for name, estimator in models.items():
    pipeline = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    print(f"\n{name}")
    print(classification_report(y_test, predictions, zero_division=0))
    print("ROC-AUC:", round(roc_auc_score(y_test, probabilities), 3))
    ConfusionMatrixDisplay.from_predictions(y_test, predictions)
    plt.title(f"{name} — synthetic demo data")
    plt.show()


## 5. Conclusions and next steps

- Treat every pattern and metric here as illustrative because the data and target are synthetic.
- Replace the CSV only with a properly licensed and documented dataset.
- For a meaningful project, document provenance, validate data quality, test generalization, assess fairness, and explain model limitations.
- Do not use this template to make real lending decisions.